# Online SAC with GeneticBatch fallback

This notebook runs or loads one selected test stream and shows when packet-loss monitoring transfers control between SAC and GA. SAC stays frozen during normal control and updates only while GA is the fallback.

In [ ]:
from pathlib import Path
import csv
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_FILE = PROJECT_ROOT / 'outputs' / 'online' / 'test_stream_results.csv'
SWITCH_FILE = PROJECT_ROOT / 'outputs' / 'online' / 'test_stream_switches.csv'
PRETRAINED_MODEL = PROJECT_ROOT / 'outputs' / 'models' / 'sac' / 'sac_pretrained_final.zip'
ADAPTED_MODEL = PROJECT_ROOT / 'outputs' / 'models' / 'sac_online' / 'sac_adapted_final.zip'
MODEL = ADAPTED_MODEL if ADAPTED_MODEL.exists() else PRETRAINED_MODEL

print('Project:', PROJECT_ROOT)
print('Model:', MODEL)

In [ ]:
RUN_EVALUATION = False
ENABLE_FALLBACK_FINETUNING = True
TEST_DATASET = 'test_slow_mix'
MAX_TIMESTEPS = 400

LOSS_WINDOW = 100
CHECK_EVERY = 20
FALLBACK_LOSS_RATE = 0.05
RECOVERY_LOSS_RATE = 0.02

if RUN_EVALUATION:
    command = [
        sys.executable, str(PROJECT_ROOT / 'src' / 'online_hybrid.py'),
        '--split', 'test',
        '--dataset', TEST_DATASET,
        '--max-timesteps', str(MAX_TIMESTEPS),
        '--checkpoint', str(MODEL),
        '--results-file', str(RESULTS_FILE),
        '--switch-file', str(SWITCH_FILE),
        '--loss-window', str(LOSS_WINDOW),
        '--check-every', str(CHECK_EVERY),
        '--fallback-loss-rate', str(FALLBACK_LOSS_RATE),
        '--recovery-loss-rate', str(RECOVERY_LOSS_RATE),
    ]
    if not ENABLE_FALLBACK_FINETUNING:
        command.append('--no-train')
    else:
        command.extend([
            '--adapted-model',
            str(PROJECT_ROOT / 'outputs' / 'models' / 'sac_online' / 'sac_test_stream_adapted.zip'),
        ])
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
    subprocess.run(command, cwd=PROJECT_ROOT, env=environment, check=True)

if not RESULTS_FILE.exists():
    raise FileNotFoundError('Run the evaluation cell by setting RUN_EVALUATION = True.')

In [ ]:
with RESULTS_FILE.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    for field in ('release_time', 'reward', 'latency', 'total_system_energy'):
        row[field] = float(row[field])
    row['rolling_loss_rate'] = (
        float(row['rolling_loss_rate']) if row['rolling_loss_rate'] else np.nan
    )
    for field in ('transmission_loss_event', 'packet_lost', 'deadline_missed'):
        row[field] = row[field].lower() == 'true'

times = np.array([row['release_time'] for row in rows])
weather_names = [row['scenario'] for row in rows]
weather_code = {'BASE': 0, 'RAIN': 1, 'SNOW': 2, 'FOG': 3}
weather = np.array([weather_code[name] for name in weather_names])
ga_active = np.array([row['controller_source'] == 'GA' for row in rows], dtype=float)
rolling_loss = np.array([row['rolling_loss_rate'] for row in rows])
reward = np.array([row['reward'] for row in rows])
latency = np.array([row['latency'] for row in rows])
energy = np.array([row['total_system_energy'] for row in rows])
packet_lost = np.array([row['packet_lost'] for row in rows], dtype=int)
deadline_missed = np.array([row['deadline_missed'] for row in rows], dtype=int)
switches = [row for row in rows if row['switch_event']]

print('Tasks:', len(rows))
print('GA-controlled tasks:', int(ga_active.sum()))
print('Final packet losses:', int(packet_lost.sum()))
print('Deadline misses:', int(deadline_missed.sum()))
print('Mean latency:', round(float(latency.mean()), 4))
print('Mean total energy:', round(float(energy.mean()), 4))
print('Switches:', [(row['switch_event'], row['release_time']) for row in switches])

In [ ]:
def rolling_mean(values, window=100):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return np.full(len(values), np.nan)
    result = np.full(len(values), np.nan)
    result[window - 1:] = np.convolve(values, np.ones(window) / window, mode='valid')
    return result

figure, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)
figure.suptitle(f'Online hybrid controller: {rows[0]["dataset"]}', fontsize=15)

axes[0, 0].step(times, weather, where='post', color='#287271')
axes[0, 0].set_yticks(list(weather_code.values()), list(weather_code.keys()))
axes[0, 0].set_title('Weather stream')

axes[0, 1].fill_between(times, 0, ga_active, step='post', color='#E07A5F', alpha=0.8)
axes[0, 1].set_yticks([0, 1], ['SAC', 'GA'])
axes[0, 1].set_title('Controller source')

axes[1, 0].plot(times, rolling_loss, color='#C44536', label='Rolling loss-event rate')
axes[1, 0].axhline(FALLBACK_LOSS_RATE, color='#9B2226', linestyle='--', label='Fallback threshold')
axes[1, 0].axhline(RECOVERY_LOSS_RATE, color='#2A9D8F', linestyle='--', label='Recovery threshold')
axes[1, 0].set_ylim(bottom=0)
axes[1, 0].set_title('Packet-loss monitor')
axes[1, 0].legend()

axes[1, 1].plot(times, rolling_mean(reward), color='#264653')
axes[1, 1].set_title('Rolling mean reward')

axes[2, 0].plot(times, rolling_mean(latency), color='#3A86FF', label='Latency')
energy_axis = axes[2, 0].twinx()
energy_axis.plot(times, rolling_mean(energy), color='#F4A261', label='Energy')
axes[2, 0].set_ylabel('Latency')
energy_axis.set_ylabel('Total system energy')
axes[2, 0].set_title('Rolling latency and energy')

axes[2, 1].plot(times, np.cumsum(packet_lost), color='#D62828', label='Packet losses')
axes[2, 1].plot(times, np.cumsum(deadline_missed), color='#6A4C93', label='Deadline misses')
axes[2, 1].set_title('Cumulative failures')
axes[2, 1].legend()

for axis in axes.flat:
    for row in switches:
        axis.axvline(row['release_time'], color='black', alpha=0.25, linewidth=1)
    axis.grid(alpha=0.2)
    axis.set_xlabel('Simulation time (s)')

plt.tight_layout()
plt.show()